# 1. Mã và thời gian: Du lieu cua VCB

In [2]:
# Import các thư viện
from ssi_fc_data import fc_md_client, model
import pandas as pd 
import json
import configDataSSI

# Create a Market Data Client
symbol = 'VCB'
from_date = "01/01/2024"
to_date = "11/06/2024"

# 2. Load data

In [4]:
client = fc_md_client.MarketDataClient(configDataSSI)

req = model.daily_ohlc(symbol, from_date, to_date, 1, 500, True)

data_dict = client.daily_ohlc(configDataSSI, req)

# Access the list of dictionaries in the "data" field
data_list = data_dict['data']

# Convert the list of dictionaries into a DataFrame
data = pd.DataFrame(data_list, columns=['TradingDate', 'Open', 'High', 'Low', 'Close', 'Volume'])
data = data.rename(columns={'TradingDate': 'Datetime'})

# Print or work with the DataFrame
data

,Datetime,Open,High,Low,Close,Volume
0,02/01/2024,82900,83600,82200,83500,1785800
1,03/01/2024,83500,84500,82800,84500,1373000
2,04/01/2024,84500,86200,84000,85900,2657900
3,05/01/2024,85900,86200,85700,86200,1180300
4,08/01/2024,86300,86800,86300,86800,1607800
...,...,...,...,...,...,...
102,05/06/2024,88700,89500,88600,88800,1882800
103,06/06/2024,88900,89700,88800,89000,1699700
104,07/06/2024,89100,89700,88500,88500,1129500
105,10/06/2024,88900,89200,87900,88000,2641600


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Datetime  107 non-null    object
 1   Open      107 non-null    object
 2   High      107 non-null    object
 3   Low       107 non-null    object
 4   Close     107 non-null    object
 5   Volume    107 non-null    object
dtypes: object(6)
memory usage: 5.1+ KB


# 3. Chien luoc

In [6]:
import pandas as pd
# data o tren

data['Datetime'] = pd.to_datetime(data['Datetime'], dayfirst=True)
data.set_index('Datetime', inplace=True)

# Giả sử 'data' là DataFrame của bạn với dữ liệu lịch sử giá cổ phiếu
data['Open'] = pd.to_numeric(data['Open'], errors='coerce')
data['High'] = pd.to_numeric(data['High'], errors='coerce')
data['Low'] = pd.to_numeric(data['Low'], errors='coerce')
data['Close'] = pd.to_numeric(data['Close'], errors='coerce')
data['Volume'] = pd.to_numeric(data['Volume'], errors='coerce')

# Định nghĩa hàm để kiểm tra nến Doji chân dài
def is_long_legged_doji(row):
    body_range = abs(row['Close'] - row['Open']) # Doji khong phan biet Open > Close hay Close > Open
    upper_shadow = row['High'] - max(row['Open'], row['Close'])
    lower_shadow = min(row['Open'], row['Close']) - row['Low']
    # Điều chỉnh ngưỡng này theo dữ liệu cụ thể của bạn
    doji_threshold = 0.1 / 100 * row['Close']
    return body_range <= doji_threshold and upper_shadow >= 2 * body_range and lower_shadow >= 2 * body_range

# Định nghĩa hàm để kiểm tra nến tăng
def is_bullish_candle(current_row, previous_row):
    return (current_row['Close'] > current_row['Open'] and
            current_row['Close'] > previous_row['Close'] and
            previous_row['Close'] <= previous_row['Open'])

# Định nghĩa hàm để kiểm tra nến giảm
def is_bearish_candle(current_row, previous_row):
    return (current_row['Close'] < current_row['Open'] and
            current_row['Close'] < previous_row['Close'] and
            previous_row['Close'] >= previous_row['Open'])

data['Buy_Signal'] = False
data['Sell_Signal'] = False 
    
for i in range(0, len(data)): # Chi lay 2 nen
    current_row = data.iloc[i]
    previous_row = data.iloc[i - 1]
    
    # Kiểm tra nến hiện tại có phải là nến tăng và nếu nến trước đó là nến Doji chân dài
    if is_bullish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        # Nếu thỏa mãn cả ba điều kiện, thêm ngày vào danh sách tín hiệu mua
        data.at[current_row.name, 'Buy_Signal'] = True

    if is_bearish_candle(current_row, previous_row) and is_long_legged_doji(previous_row):
        data.at[current_row.name, 'Sell_Signal'] = True

data

,Open,High,Low,Close,Volume,Buy_Signal,Sell_Signal
Datetime,,,,,,,
2024-01-02,82900,83600,82200,83500,1785800,False,False
2024-01-03,83500,84500,82800,84500,1373000,False,False
2024-01-04,84500,86200,84000,85900,2657900,False,False
2024-01-05,85900,86200,85700,86200,1180300,False,False
2024-01-08,86300,86800,86300,86800,1607800,False,False
...,...,...,...,...,...,...,...
2024-06-05,88700,89500,88600,88800,1882800,False,False
2024-06-06,88900,89700,88800,89000,1699700,False,False
2024-06-07,89100,89700,88500,88500,1129500,False,False


In [7]:
data.to_csv('data.csv')

# 4. Backtest chien luoc dua vao Data tren

In [8]:
def backtest(data, initial_capital, shares_per_signal): # Chung khoan
        import pandas as pd
        import matplotlib.pyplot as plt
        import plotly.graph_objects as go
        import plotly.express as px

        data['Close'] = data['Close'].ffill()   
        data.dropna(subset=['Close'], inplace=True)
        
        capital = initial_capital
        shares_held = 0

        # Xác định vị thế mua/ bán
        data['Position_Buy'] = data['Buy_Signal'].shift(1)
        data['Position_Sell'] = data['Sell_Signal'].shift(1)

        data['Trade_Action'] = ''
        data['Capital'] = capital
        data['Shares_Held'] = shares_held

        data['Capital'] = data['Capital'].astype(float)
        data['Shares_Held'] = data['Shares_Held'].astype(float)

        # Lặp qua mỗi hàng trong DataFrame
        for index, row in data.iterrows():
            # Nếu có tín hiệu mua và có đủ vốn để mua
            if row['Position_Buy'] == 1 and capital >= row['Close'] * shares_per_signal and row['Trade_Action'] == '':
                # Mua cổ phiếu và cập nhật vốn và số cổ phiếu được giữ
                data.at[index, 'Trade_Action'] = 'Buy'
                capital -= row['Close'] * shares_per_signal
                data.at[index, 'Capital'] = capital
                shares_held += shares_per_signal
                data.at[index, 'Shares_Held'] = shares_held
            elif row['Position_Sell'] == 1 and shares_held > 0 and row['Trade_Action'] == '':
                data.at[index, 'Trade_Action'] = 'Sell'
                capital += row['Close'] * shares_held
                data.at[index, 'Capital'] = capital
                shares_held = 0
                data.at[index, 'Shares_Held'] = shares_held  # Giảm số lượng cổ phiếu 0
            else:
                data.at[index, 'Capital'] = capital
                data.at[index, 'Shares_Held'] = shares_held

            # Cập nhật giá trị hiện tại của vốn dựa trên số cổ phiếu đang giữ và giá đóng cửa hiện tại
            current_value = capital + shares_held * row['Close']

        # Ngày vào lệnh
        first_entry_date = data[data['Position_Buy'] == 1].index.min()
        # Tính lợi nhuận
        profit = current_value - initial_capital
        # Tính lợi nhuận thị trường
        market_return = (data['Close'].iloc[-1] - data['Close'].iloc[0]) / data['Close'].iloc[0]
        # Tính lợi nhuận chiến lược
        strategy_return = (current_value - initial_capital) / initial_capital

        print(f"Ngày vào lệnh đầu tiên: {first_entry_date}")
        print(f'Tổng lợi nhuận: {profit}')
        print(f'Tổng giá trị tài khoản: {current_value}')
        print(f'Lợi nhuận thị trường: {market_return * 100}%')
        print(f'Lợi nhuận chiến lược: {strategy_return * 100}%')

        # Tính toán lợi nhuận thị trường và chiến lược
        data['Market_Return'] = data['Close'].pct_change()
        data['Cumulative_Market_Returns'] = (1 + data['Market_Return']).cumprod()

        # Tính toán lợi nhuận lũy kế từ chiến lược
        data['Strategy_Value'] = data['Capital'] + data['Shares_Held'] * data['Close']
        data['Cumulative_Strategy_Returns'] = data['Strategy_Value'] / initial_capital       

        # Tạo biểu đồ sử dụng Plotly
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=data.index, y=data['Cumulative_Market_Returns'], mode='lines', name='Market Returns'))
        fig.add_trace(go.Scatter(x=data.index, y=data['Cumulative_Strategy_Returns'], mode='lines', name='Strategy Returns'))

        fig.update_layout(
            title='Comparison of Cumulative Returns: Market vs Strategy',
            xaxis_title='Date',
            yaxis_title='Cumulative Returns',
        )
        fig.show()

        return data

# 4. Chay backtest

In [9]:
resultBacktest = backtest(data, 100000000.0, 10)

Ngày vào lệnh đầu tiên: 2024-03-21 00:00:00
Tổng lợi nhuận: -101000.0
Tổng giá trị tài khoản: 99899000.0
Lợi nhuận thị trường: 4.431137724550898%
Lợi nhuận chiến lược: -0.101%


In [10]:
resultBacktest.to_csv('resultBacktest.csv')
resultBacktest

,Open,High,Low,Close,Volume,Buy_Signal,Sell_Signal,Position_Buy,Position_Sell,Trade_Action,Capital,Shares_Held,Market_Return,Cumulative_Market_Returns,Strategy_Value,Cumulative_Strategy_Returns
Datetime,,,,,,,,,,,,,,,,
2024-01-02,82900,83600,82200,83500,1785800,False,False,NaN,NaN,,100000000.0,0.0,NaN,NaN,100000000.0,1.00000
2024-01-03,83500,84500,82800,84500,1373000,False,False,False,False,,100000000.0,0.0,0.011976,1.011976,100000000.0,1.00000
2024-01-04,84500,86200,84000,85900,2657900,False,False,False,False,,100000000.0,0.0,0.016568,1.028743,100000000.0,1.00000
2024-01-05,85900,86200,85700,86200,1180300,False,False,False,False,,100000000.0,0.0,0.003492,1.032335,100000000.0,1.00000
2024-01-08,86300,86800,86300,86800,1607800,False,False,False,False,,100000000.0,0.0,0.006961,1.039521,100000000.0,1.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-06-05,88700,89500,88600,88800,1882800,False,False,False,False,,98155000.0,20.0,0.001127,1.063473,99931000.0,0.99931
2024-06-06,88900,89700,88800,89000,1699700,False,False,False,False,,98155000.0,20.0,0.002252,1.065868,99935000.0,0.99935
2024-06-07,89100,89700,88500,88500,1129500,False,False,False,False,,98155000.0,20.0,-0.005618,1.059880,99925000.0,0.99925
